**Run note:** execute this notebook's first setup/code cell before any later cells. Each notebook is designed to run independently and re-detect the dataset path on its own.

# 11 — Explainability

Responsible AI layer:
- GradCAM on CLIP vision encoder (image regions that drive prediction)
- Cross-attention weight visualization
- Per-demographic-group fairness audit
- Confidence calibration (reliability diagram)
- Token-level attribution (text)

In [1]:
import os
import json
import re
import time
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
from PIL import Image
from sklearn.metrics import f1_score, roc_auc_score, accuracy_score
from sklearn.calibration import calibration_curve
from torch.utils.data import Dataset, DataLoader
from transformers import CLIPModel, CLIPProcessor

ON_KAGGLE = Path("/kaggle/input").is_dir()
JSONL_CANDIDATES = {
    "train": ["train.jsonl"],
    "dev": ["dev.jsonl", "dev_seen.jsonl", "dev_unseen.jsonl"],
    "test": ["test.jsonl", "test_seen.jsonl", "test_unseen.jsonl"],
}
IMAGE_DIR_CANDIDATES = ("img", "images")


def _has_image_dir(path: Path) -> bool:
    return any((path / name).is_dir() for name in IMAGE_DIR_CANDIDATES)


def _has_any_jsonl(path: Path, names) -> bool:
    return any((path / name).is_file() for name in names)


def _looks_like_dataset_root(path: Path) -> bool:
    return path.is_dir() and _has_image_dir(path) and _has_any_jsonl(path, JSONL_CANDIDATES["train"])


def detect_data_dir():
    for env_name in ("KAGGLE_DATA_DIR", "META_HATEFUL_MEME_DATA_DIR"):
        env_dir = os.environ.get(env_name, "").strip()
        if env_dir and _looks_like_dataset_root(Path(env_dir)):
            return Path(env_dir), f"env:{env_name}"

    kaggle_input = Path("/kaggle/input")
    for default_sub in (
        "meta-hateful-meme-detection/data",
        "datasets/muddybuddy/meta-hateful-meme-detection/data",
    ):
        default_candidate = kaggle_input / default_sub
        if _looks_like_dataset_root(default_candidate):
            return default_candidate, f"default:{default_candidate}"

    if ON_KAGGLE:
        for train_jsonl in sorted(kaggle_input.rglob("train.jsonl")):
            candidate = train_jsonl.parent
            if _looks_like_dataset_root(candidate):
                return candidate, f"auto:{candidate}"

        for candidate in sorted(kaggle_input.rglob("*")):
            if candidate.is_dir() and _looks_like_dataset_root(candidate):
                return candidate, f"auto:{candidate}"

    for candidate in (Path.cwd() / "data", Path.cwd().parent / "data", Path.cwd(), Path.cwd().parent):
        if _looks_like_dataset_root(candidate):
            return candidate, f"local:{candidate}"

    return None, "not-found"


def resolve_split(base_dir, names):
    base_dir = Path(base_dir)
    for name in names:
        path = base_dir / name
        if path.is_file():
            return path
    for name in names:
        matches = sorted(base_dir.rglob(name))
        if matches:
            return matches[0]
    return None


DATA_DIR, data_source = detect_data_dir()
if DATA_DIR is None:
    raise FileNotFoundError(
        "Dataset not found. Set KAGGLE_DATA_DIR or META_HATEFUL_MEME_DATA_DIR to the folder containing train.jsonl and img/."
    )

IMG_DIR = next((DATA_DIR / name for name in IMAGE_DIR_CANDIDATES if (DATA_DIR / name).is_dir()), None)
DEV_PATH = resolve_split(DATA_DIR, JSONL_CANDIDATES["dev"])
LOCAL_OUTPUT_DIR = Path.cwd() / "outputs"
OUTPUT_DIR = Path("/kaggle/working") if ON_KAGGLE else LOCAL_OUTPUT_DIR
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

if DEV_PATH is None:
    raise FileNotFoundError(f"Expected dev split under {DATA_DIR}")

CFG = {
    "clip_model": "openai/clip-vit-base-patch32",
    "batch_size": 32,
    "embed_dim": 512,
    "num_heads": 4,
    "dropout": 0.3,
    "max_text_len": 77,
}

# --- Pre-cache CLIP model locally ---
_CLIP_CACHE = OUTPUT_DIR / "hf_clip_cache" / "clip-vit-base-patch32"
_CLIP_CACHE.mkdir(parents=True, exist_ok=True)

def _is_valid_b32_cache(cache_dir):
    """Check cache has config.json + tokenizer_config.json AND is actually ViT-B/32 (projection_dim=512)."""
    cfg_path = cache_dir / "config.json"
    tok_path = cache_dir / "tokenizer_config.json"
    if not cfg_path.exists() or not tok_path.exists():
        return False
    try:
        with open(cfg_path, encoding="utf-8") as f:
            cfg_data = json.load(f)
        proj_dim = cfg_data.get("projection_dim", -1)
        if proj_dim != CFG["embed_dim"]:
            print(f"WARNING: Cache at {cache_dir} has projection_dim={proj_dim}, expected {CFG['embed_dim']}. Re-downloading.", flush=True)
            return False
        return True
    except Exception:
        return False

def _find_clip_source():
    """Search for valid ViT-B/32 cache: own cache → attached inputs → download."""
    if _is_valid_b32_cache(_CLIP_CACHE):
        return str(_CLIP_CACHE), "own-cache"
    if ON_KAGGLE and Path("/kaggle/input").is_dir():
        for cfg_file in sorted(Path("/kaggle/input").rglob("clip-vit-base-patch32/config.json")):
            parent = cfg_file.parent
            if _is_valid_b32_cache(parent):
                return str(parent), f"attached:{parent}"
    return None, "download-needed"

CLIP_MODEL_SOURCE, _clip_src = _find_clip_source()
if CLIP_MODEL_SOURCE is not None:
    print(f"Using cached CLIP from {CLIP_MODEL_SOURCE} ({_clip_src})", flush=True)
else:
    print("Downloading CLIP model + processor to local cache...", flush=True)
    # Clear any bad cache first
    import shutil
    if _CLIP_CACHE.exists():
        shutil.rmtree(str(_CLIP_CACHE))
        _CLIP_CACHE.mkdir(parents=True, exist_ok=True)
    t0 = time.perf_counter()
    _m = CLIPModel.from_pretrained(CFG["clip_model"])
    _m.save_pretrained(str(_CLIP_CACHE))
    del _m
    _p = CLIPProcessor.from_pretrained(CFG["clip_model"])
    _p.save_pretrained(str(_CLIP_CACHE))
    del _p
    torch.cuda.empty_cache()
    CLIP_MODEL_SOURCE = str(_CLIP_CACHE)
    print(f"CLIP cached in {time.perf_counter() - t0:.1f}s to {CLIP_MODEL_SOURCE}", flush=True)

print(f"Using DATA_DIR : {DATA_DIR}", flush=True)
print(f"Using IMG_DIR  : {IMG_DIR}", flush=True)
print(f"Using source   : {data_source}", flush=True)
print(f"Output dir     : {OUTPUT_DIR}", flush=True)
print(f"Device         : {DEVICE}", flush=True)


def load_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return pd.DataFrame([json.loads(l) for l in f])


def clean_text(text):
    if not isinstance(text, str):
        return "[no text]"
    return re.sub(
        r"\s+",
        " ",
        re.sub(r"@\w+", " ", re.sub(r"https?://\S+", " ", text)),
    ).strip() or "[no text]"


dev_df = load_jsonl(str(DEV_PATH))
dev_df["clean_text"] = dev_df["text"].apply(clean_text)
print(f"Dev: {len(dev_df)} samples", flush=True)

Using cached CLIP from /kaggle/input/notebooks/muddybuddy/10-evaluation/hf_clip_cache/clip-vit-base-patch32 (attached:/kaggle/input/notebooks/muddybuddy/10-evaluation/hf_clip_cache/clip-vit-base-patch32)
Using DATA_DIR : /kaggle/input/datasets/muddybuddy/meta-hateful-meme-detection/data
Using IMG_DIR  : /kaggle/input/datasets/muddybuddy/meta-hateful-meme-detection/data/img
Using source   : default:/kaggle/input/datasets/muddybuddy/meta-hateful-meme-detection/data
Output dir     : /kaggle/working
Device         : cuda
Dev: 500 samples


In [2]:
from pathlib import Path


def resolve_image_path(data_dir, image_ref):
    data_dir = Path(data_dir)
    image_ref = Path(str(image_ref))

    candidates = []
    if image_ref.is_absolute():
        candidates.append(image_ref)

    candidates.extend([
        data_dir / image_ref,
        data_dir.parent / image_ref,
    ])

    if image_ref.parts:
        if image_ref.parts[0] in {"img", "images"} and len(image_ref.parts) > 1:
            stripped = Path(*image_ref.parts[1:])
            candidates.extend([
                data_dir / stripped,
                data_dir.parent / stripped,
            ])
        elif image_ref.parts[0] not in {"img", "images"}:
            candidates.extend([
                data_dir / "img" / image_ref,
                data_dir / "images" / image_ref,
                data_dir.parent / "img" / image_ref,
                data_dir.parent / "images" / image_ref,
            ])

    seen = set()
    for candidate in candidates:
        key = str(candidate)
        if key in seen:
            continue
        seen.add(key)
        if candidate.exists():
            return candidate

    raise FileNotFoundError(f"Could not find image '{image_ref}' relative to {data_dir}")

In [3]:
# ── Inline model (same as 08/09) ─────────────────────────────────────

def _ensure_tensor(x):
    if isinstance(x, torch.Tensor):
        return x
    if hasattr(x, "last_hidden_state"):
        return x.last_hidden_state[:, 0]
    if hasattr(x, "pooler_output"):
        return x.pooler_output
    return x[0] if isinstance(x, (tuple, list)) else x

class CLIPEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.clip = CLIPModel.from_pretrained(CLIP_MODEL_SOURCE, local_files_only=True)
        for p in self.clip.parameters(): p.requires_grad_(False)
    def forward(self, pv, ids, mask):
        i = F.normalize(_ensure_tensor(self.clip.get_image_features(pixel_values=pv)), dim=-1)
        t = F.normalize(_ensure_tensor(self.clip.get_text_features(input_ids=ids, attention_mask=mask)), dim=-1)
        return i, t

class CrossAttentionFusion(nn.Module):
    def __init__(self, d=512, h=4):
        super().__init__()
        self.i2t = nn.MultiheadAttention(d, h, batch_first=True)
        self.t2i = nn.MultiheadAttention(d, h, batch_first=True)
        self.ni  = nn.LayerNorm(d); self.nt = nn.LayerNorm(d)
    def forward(self, i, t):
        is_ = i.unsqueeze(1); ts = t.unsqueeze(1)
        ic, ia = self.i2t(is_, ts, ts); tc, ta = self.t2i(ts, is_, is_)
        return torch.cat([self.ni(i+ic.squeeze(1)), self.nt(t+tc.squeeze(1))], -1), ia, ta

class ClassificationHead(nn.Module):
    def __init__(self, d=1024):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d,256),nn.GELU(),nn.Dropout(0.3),
                                  nn.Linear(256,128),nn.GELU(),nn.Dropout(0.3),nn.Linear(128,2))
    def forward(self, x): return self.net(x)

class HatefulMemeClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = CLIPEncoder(); self.fusion = CrossAttentionFusion(); self.head = ClassificationHead()
    def forward(self, pv, ids, mask):
        i,t = self.encoder(pv,ids,mask); f,ia,ta = self.fusion(i,t); return self.head(f),ia,ta

processor = CLIPProcessor.from_pretrained(CLIP_MODEL_SOURCE, use_fast=False, local_files_only=True)

# --- Checkpoint search ---
def find_checkpoint(filename):
    candidates = [
        OUTPUT_DIR / filename,
        LOCAL_OUTPUT_DIR / filename,
        LOCAL_OUTPUT_DIR / "baselines" / filename,
    ]
    if ON_KAGGLE:
        candidates.append(Path("/kaggle/working") / filename)
        for pt in sorted(Path("/kaggle/input").rglob(filename)):
            candidates.append(pt)
    for c in candidates:
        if c.exists():
            return c
    return None

model = HatefulMemeClassifier()

# Verify CLIP output dimension matches expected embed_dim
with torch.no_grad():
    _dummy_pv = torch.randn(1, 3, 224, 224)
    _dummy_ids = torch.zeros(1, 10, dtype=torch.long)
    _dummy_mask = torch.ones(1, 10, dtype=torch.long)
    _i, _t = model.encoder(_dummy_pv, _dummy_ids, _dummy_mask)
    _clip_dim = _i.shape[-1]
    if _clip_dim != CFG["embed_dim"]:
        raise RuntimeError(
            f"CLIP output dim is {_clip_dim} but CFG expects {CFG['embed_dim']}. "
            f"CLIP_MODEL_SOURCE={CLIP_MODEL_SOURCE} is likely pointing to the wrong model. "
            f"Delete the cache at {_CLIP_CACHE} and re-run cell 3."
        )
    print(f"CLIP output dim verified: {_clip_dim}", flush=True)
    del _dummy_pv, _dummy_ids, _dummy_mask, _i, _t

model = model.to(DEVICE)
ckpt = find_checkpoint("cross_attention_best.pt")
if ckpt is not None:
    model.load_state_dict(torch.load(str(ckpt), map_location=DEVICE, weights_only=False))
    print(f"Checkpoint loaded from {ckpt}", flush=True)
else:
    print("WARNING: checkpoint not found — using random weights for demonstration.", flush=True)
model.eval()
print("Model ready.", flush=True)

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

RuntimeError: CLIP output dim is 768 but CFG expects 512. CLIP_MODEL_SOURCE=/kaggle/input/notebooks/muddybuddy/10-evaluation/hf_clip_cache/clip-vit-base-patch32 is likely pointing to the wrong model. Delete the cache at /kaggle/working/hf_clip_cache/clip-vit-base-patch32 and re-run cell 3.

In [ ]:
# ── 11.1 GradCAM on CLIP vision encoder ──────────────────────────────
class CLIPGradCAM:
    """
    Attaches hooks to the last transformer layer of CLIP's vision encoder.
    Computes GradCAM heatmap by backpropagating the hateful class score.
    """
    def __init__(self, model):
        self.model      = model
        self.activations = None
        self.gradients   = None
        # Hook the last layer of CLIP's vision transformer
        target_layer = model.encoder.clip.vision_model.encoder.layers[-1]
        target_layer.register_forward_hook(self._save_activation)
        target_layer.register_full_backward_hook(self._save_gradient)

    def _save_activation(self, module, inp, output):
        del module, inp  # required by hook signature
        self.activations = output[0].detach()  # (B, seq_len, d)

    def _save_gradient(self, module, grad_in, grad_out):
        del module, grad_in  # required by hook signature
        self.gradients = grad_out[0].detach()  # (B, seq_len, d)

    def generate(self, pixel_values, input_ids, attention_mask, target_class=1):
        self.model.zero_grad()
        pixel_values = pixel_values.requires_grad_(True)
        logits, _, _ = self.model(pixel_values, input_ids, attention_mask)
        score        = logits[0, target_class]
        score.backward()

        # Pool gradients across feature dimension
        weights = self.gradients.mean(dim=-1, keepdim=True)  # (B, seq_len, 1)
        cam     = (weights * self.activations).sum(-1)[0]    # (seq_len,)
        cam     = F.relu(cam)                                 # keep positive

        # Remove [CLS] token, reshape to patch grid
        # ViT-B/32: 224/32 = 7 → 7x7 = 49 patches; ViT-B/16: 14x14 = 196
        patch_tokens = cam[1:]
        grid_size = int(patch_tokens.shape[0] ** 0.5)
        cam = patch_tokens.reshape(grid_size, grid_size)
        cam = cam.cpu().numpy()
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam

gradcam = CLIPGradCAM(model)
print("GradCAM hooks attached.", flush=True)

In [ ]:
# ── Show GradCAM on 4 dev samples ─────────────────────────────────────
samples = dev_df[dev_df["label"] == 1].head(4)

fig, axes = plt.subplots(4, 3, figsize=(12, 14))
fig.suptitle("GradCAM Explainability — Hateful Samples", fontsize=13, fontweight="bold")

for row_idx, (_, row) in enumerate(samples.iterrows()):
    img_path = resolve_image_path(DATA_DIR, row["img"])
    try:
        pil_img = Image.open(img_path).convert("RGB")
    except Exception:
        pil_img = Image.new("RGB", (224, 224), 128)

    text = str(row["clean_text"])
    enc  = processor(text=[text], images=pil_img, return_tensors="pt",
                      padding="max_length", max_length=77, truncation=True)

    pv   = enc["pixel_values"].to(DEVICE)
    ids  = enc["input_ids"].to(DEVICE)
    mask = enc["attention_mask"].to(DEVICE)

    prob = 0.0
    try:
        cam = gradcam.generate(pv, ids, mask)
        heatmap_resized = np.array(Image.fromarray((cm.jet(cam)[:, :, :3] * 255).astype(np.uint8)).resize(
            pil_img.size, Image.BILINEAR))
        overlay = (0.5 * np.array(pil_img).astype(float) + 0.5 * heatmap_resized).astype(np.uint8)
    except Exception as e:
        print(f"  GradCAM failed for sample {row_idx}: {e}", flush=True)
        cam     = np.zeros((7, 7))
        overlay = np.array(pil_img)

    try:
        with torch.no_grad():
            logits, _, _ = model(pv, ids, mask)
            prob = torch.softmax(logits, -1)[0, 1].item()
    except Exception as e:
        print(f"  Inference failed for sample {row_idx}: {e}", flush=True)

    axes[row_idx][0].imshow(pil_img); axes[row_idx][0].set_title("Original", fontsize=8); axes[row_idx][0].axis("off")
    axes[row_idx][1].imshow(cam, cmap="jet"); axes[row_idx][1].set_title("GradCAM Heatmap", fontsize=8); axes[row_idx][1].axis("off")
    axes[row_idx][2].imshow(overlay); axes[row_idx][2].set_title(f"Overlay (P(hate)={prob:.2f})", fontsize=8); axes[row_idx][2].axis("off")

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "gradcam_examples.png"), dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
#  10.2 Per-demographic-group fairness audit
class MemeDataset(Dataset):
    def __init__(self, df, data_dir, processor):
        self.df = df.reset_index(drop=True); self.data_dir = data_dir; self.processor = processor
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        try: img = Image.open(resolve_image_path(self.data_dir, row["img"])).convert("RGB")
        except: img = Image.new("RGB", (224,224), 128)
        enc = self.processor(text=[str(row.get("clean_text", row["text"]))], images=img,
                               return_tensors="pt", padding="max_length", max_length=77, truncation=True)
        return {"pixel_values": enc["pixel_values"].squeeze(0),
                "input_ids": enc["input_ids"].squeeze(0),
                "attention_mask": enc["attention_mask"].squeeze(0),
                "label": torch.tensor(int(row["label"]), dtype=torch.long),
                "text": str(row.get("clean_text", row["text"]))}

DEMOGRAPHICS = {
    "race"    : r"\b(black|white|asian|african|hispanic|arab|jews|jewish)\b",
    "gender"  : r"\b(woman|women|man|men|female|male|girl|boy)\b",
    "religion": r"\b(muslim|christian|jew|islamic|hindu|buddhist|christ)\b",
    "lgbtq"   : r"\b(gay|lesbian|trans|lgbt|queer)\b",
}

# Run inference on full dev set
dev_loader = DataLoader(MemeDataset(dev_df, DATA_DIR, processor),
                         batch_size=32, shuffle=False, num_workers=0)

all_probs, all_preds, all_labs, all_texts = [], [], [], []
model.eval()
with torch.no_grad():
    for batch in dev_loader:
        pv   = batch["pixel_values"].to(DEVICE)
        ids  = batch["input_ids"].to(DEVICE)
        mask = batch["attention_mask"].to(DEVICE)
        logits, _, _ = model(pv, ids, mask)
        probs = torch.softmax(logits, -1)[:, 1].cpu().numpy()
        preds = logits.argmax(-1).cpu().numpy()
        all_probs.extend(probs); all_preds.extend(preds)
        all_labs.extend(batch["label"].numpy()); all_texts.extend(batch["text"])

dev_df["pred"]  = all_preds
dev_df["prob"]  = all_probs

bias_rows = []
for group, pattern in DEMOGRAPHICS.items():
    mask_grp = dev_df["clean_text"].str.lower().str.contains(pattern, regex=True)
    subset   = dev_df[mask_grp]
    if len(subset) < 5:
        continue
    bias_rows.append({
        "Group"       : group,
        "N"           : len(subset),
        "Accuracy"    : accuracy_score(subset["label"], subset["pred"]),
        "F1 (Hateful)": f1_score(subset["label"], subset["pred"], pos_label=1, zero_division=0),
        "AUROC"       : roc_auc_score(subset["label"], subset["prob"]) if subset["label"].nunique() > 1 else float("nan"),
        "FPR"         : ((subset["label"]==0) & (subset["pred"]==1)).sum() / max((subset["label"]==0).sum(), 1),
        "FNR"         : ((subset["label"]==1) & (subset["pred"]==0)).sum() / max((subset["label"]==1).sum(), 1),
    })

bias_df = pd.DataFrame(bias_rows)
print("\nFAIRNESS AUDIT  Per-Group Metrics")
print(bias_df.to_string(index=False, float_format="{:.3f}".format))

# Overall for comparison
print(f"\nOverall AUROC: {roc_auc_score(dev_df.label, dev_df.prob):.4f}")
print(f"Overall F1   : {f1_score(dev_df.label, dev_df.pred, pos_label=1):.4f}")

In [ ]:
# â”€â”€ 10.3 Demographic bias heatmap â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
if not bias_df.empty:
    plot_cols = ["Accuracy", "F1 (Hateful)", "FPR", "FNR"]
    heat_data = bias_df.set_index("Group")[plot_cols]

    fig, ax = plt.subplots(figsize=(9, 4))
    sns.heatmap(heat_data, annot=True, fmt=".3f", cmap="RdYlGn",
                linewidths=0.5, ax=ax, vmin=0, vmax=1)
    ax.set_title("Fairness Audit â€” Per-Demographic Performance", fontsize=12, fontweight="bold")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "fairness_heatmap.png"), dpi=150, bbox_inches="tight")
    plt.show()

In [ ]:
# â”€â”€ 10.4 Calibration (reliability diagram) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
frac_pos, mean_pred = calibration_curve(
    dev_df["label"], dev_df["prob"], n_bins=10, strategy="uniform"
)

fig, ax = plt.subplots(figsize=(6, 6))
ax.plot([0,1],[0,1], "k--", label="Perfect calibration")
ax.plot(mean_pred, frac_pos, "o-", color="#E91E63", lw=2, ms=8, label="Our model")
ax.fill_between(mean_pred, frac_pos, mean_pred, alpha=0.15, color="#E91E63")
ax.set_xlabel("Mean predicted probability"); ax.set_ylabel("Fraction of positives")
ax.set_title("Reliability Diagram (Calibration)", fontsize=12, fontweight="bold")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "calibration_curve.png"), dpi=150, bbox_inches="tight")
plt.show()

from sklearn.metrics import brier_score_loss
print(f"Brier Score (lower=better): {brier_score_loss(dev_df.label, dev_df.prob):.4f}")
print("Explainability notebook complete. Proceed to notebook 11 (Counterfactuals).")